# Mise en œuvre de la régression softmax à partir de zéro
:label:`sec_softmax_scratch`

Parce que la régression softmax est si fondamentale,
nous pensons que vous devriez savoir
comment la mettre en œuvre vous-même.
Ici, nous nous limitons à définir les
aspects spécifiques au softmax du modèle
et réutilisons les autres composants
de notre section sur la régression linéaire,
y compris la boucle d'entraînement.


## Le Softmax

Commençons par la partie la plus importante :
la transformation de scalaires en probabilités.
Pour un rappel, souvenez-vous de l'opération de l'opérateur somme
le long de dimensions spécifiques dans un tenseur,
comme discuté dans :numref:`subsec_lin-alg-reduction`
et :numref:`subsec_lin-alg-non-reduction`.
[**Étant donné une matrice `X`, nous pouvons faire la somme de tous les éléments (par défaut) ou seulement
des éléments d'un même axe.**]
La variable `axis` nous permet de calculer les sommes par ligne et par colonne :


In [ ]:
X = d2l.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
d2l.reduce_sum(X, 0, keepdims=True), d2l.reduce_sum(X, 1, keepdims=True)

Le calcul du softmax nécessite trois étapes :
(i) l'exponentiation de chaque terme ;
(ii) une somme sur chaque ligne pour calculer la constante de normalisation pour chaque exemple ;
(iii) la division de chaque ligne par sa constante de normalisation,
garantissant que le résultat somme à 1 :

(**
$$\mathrm{softmax}(\mathbf{X})_{ij} = \frac{\exp(\mathbf{X}_{ij})}{\sum_k \exp(\mathbf{X}_{ik})}.$$
**)

Le (logarithme du) dénominateur
est appelé la *fonction de partition* (log).
Elle a été introduite en [physique statistique](https://en.wikipedia.org/wiki/Partition_function_(statistical_mechanics))
pour sommer tous les états possibles dans un ensemble thermodynamique.
La mise en œuvre est simple :


In [ ]:
def softmax(X):
    X_exp = d2l.exp(X)
    partition = d2l.reduce_sum(X_exp, 1, keepdims=True)
    return X_exp / partition  # The broadcasting mechanism is applied here

Pour toute entrée `X`, [**nous transformons chaque élément
en un nombre non négatif.
Chaque ligne somme à 1,**]
comme requis pour une probabilité. Attention : le code ci-dessus n'est *pas* robuste face à de très grands ou de très petits arguments. Bien qu'il soit suffisant pour illustrer ce qui se passe, vous ne devriez *pas* utiliser ce code textuellement pour un usage sérieux. Les frameworks de deep learning intègrent de telles protections et nous utiliserons le softmax intégré à l'avenir.


## Le Modèle

Nous avons maintenant tout ce dont nous avons besoin
pour mettre en œuvre [**le modèle de régression softmax.**]
Comme dans notre exemple de régression linéaire,
chaque instance sera représentée
par un vecteur de longueur fixe.
Étant donné que les données brutes consistent ici
en des images de $28 \times 28$ pixels,
[**nous aplatissons chaque image,
en les traitant comme des vecteurs de longueur 784.**]
Dans les chapitres suivants, nous introduirons
les réseaux de neurones convolutionnels,
qui exploitent la structure spatiale
de manière plus satisfaisante.


Dans la régression softmax,
le nombre de sorties de notre réseau
doit être égal au nombre de classes.
(**Comme notre ensemble de données comporte 10 classes,
notre réseau a une dimension de sortie de 10.**)
Par conséquent, nos poids constituent une matrice de $784 \times 10$
plus un vecteur ligne de $1 \times 10$ pour les biais.
Comme pour la régression linéaire,
nous initialisons les poids `W`
avec un bruit gaussien.
Les biais sont initialisés à zéro.


Le code ci-dessous définit comment le réseau
projette chaque entrée vers une sortie.
Notez que nous aplatissons chaque image de $28 \times 28$ pixels du lot
en un vecteur à l'aide de `reshape`
avant de passer les données dans notre modèle.


In [ ]:
@d2l.add_to_class(SoftmaxRegressionScratch)
def forward(self, X):
    X = d2l.reshape(X, (-1, self.W.shape[0]))
    return softmax(d2l.matmul(X, self.W) + self.b)

## La Perte d'Entropie Croisée

Ensuite, nous devons mettre en œuvre la fonction de perte d'entropie croisée
(introduite dans :numref:`subsec_softmax-regression-loss-func`).
C'est peut-être la fonction de perte la plus courante
dans tout le deep learning.
À l'heure actuelle, les applications du deep learning
pouvant être facilement formulées comme des problèmes de classification
dépassent de loin celles mieux traitées comme des problèmes de régression.

Rappelez-vous que l'entropie croisée prend la log-vraisemblance négative
de la probabilité prédite affectée à la véritable étiquette.
Par souci d'efficacité, nous évitons les boucles for de Python et utilisons l'indexation à la place.
En particulier, l'encodage one-hot dans $\mathbf{y}$
nous permet de sélectionner les termes correspondants dans $\hat{\mathbf{y}}$.

Pour voir cela en action, nous [**créons des données d'exemple `y_hat`
avec 2 exemples de probabilités prédites sur 3 classes et leurs étiquettes correspondantes `y`.**]
Les bonnes étiquettes sont respectivement $0$ et $2$ (c'est-à-dire la première et la troisième classe).
[**En utilisant `y` comme indices des probabilités dans `y_hat`,**]
nous pouvons choisir les termes efficacement.


## Entraînement

Nous réutilisons la méthode `fit` définie dans :numref:`sec_linear_scratch` pour [**entraîner le modèle avec 10 époques.**]
Notez que le nombre d'époques (`max_epochs`),
la taille du mini-lot (`batch_size`),
et le taux d'apprentissage (`lr`)
sont des hyperparamètres ajustables.
Cela signifie que bien que ces valeurs ne soient pas
apprises lors de notre boucle d'entraînement principale,
elles influencent toujours les performances
de notre modèle, tant vis-à-vis de l'entraînement
que des performances de généralisation.
En pratique, vous voudrez choisir ces valeurs
en fonction de la division de *validation* des données
puis, finalement, évaluer votre modèle final
sur la division de *test*.
Comme discuté dans :numref:`subsec_generalization-model-selection`,
nous considérerons les données de test de Fashion-MNIST
comme l'ensemble de validation, rapportant ainsi
la perte de validation et la précision de validation
sur cette division.


In [ ]:
data = d2l.FashionMNIST(batch_size=256)
model = SoftmaxRegressionScratch(num_inputs=784, num_outputs=10, lr=0.1)
trainer = d2l.Trainer(max_epochs=10)
trainer.fit(model, data)

## Prédiction

Maintenant que l'entraînement est terminé,
notre modèle est prêt à [**classer quelques images.**]


In [ ]:
X, y = next(iter(data.val_dataloader()))
preds.shape

Nous sommes plus intéressés par les images que nous étiquetons *incorrectement*. Nous les visualisons en
comparant leurs étiquettes réelles
(première ligne de la sortie texte)
avec les prédictions du modèle
(deuxième ligne de la sortie texte).


In [ ]:
wrong = d2l.astype(preds, y.dtype) != y
X, y, preds = X[wrong], y[wrong], preds[wrong]
labels = [a+'\n'+b for a, b in zip(
    data.text_labels(y), data.text_labels(preds))]
data.visualize([X, y], labels=labels)

## Résumé

À présent, nous commençons à acquérir une certaine expérience
dans la résolution de problèmes de régression linéaire
et de classification.
Avec cela, nous avons atteint ce qui serait sans doute
l'état de l'art du modelage statistique des années 1960--1970.
Dans la section suivante, nous vous montrerons comment tirer parti
des frameworks de deep learning pour mettre en œuvre ce modèle
beaucoup plus efficacement.

## Exercices

1. Dans cette section, nous avons directement mis en œuvre la fonction softmax sur la base de la définition mathématique de l'opération softmax. Comme discuté dans :numref:`sec_softmax`, cela peut provoquer des instabilités numériques.
    1. Testez si `softmax` fonctionne toujours correctement si une entrée a une valeur de $100$.
    1. Testez si `softmax` fonctionne toujours correctement si la plus grande de toutes les entrées est inférieure à $-100$.
    1. Mettez en œuvre un correctif en examinant la valeur par rapport à la plus grande entrée de l'argument.
1. Mettez en œuvre une fonction `cross_entropy` qui suit la définition de la fonction de perte d'entropie croisée $\sum_i y_i \log \hat{y}_i$.
    1. Essayez-la dans l'exemple de code de cette section.
    1. Pourquoi pensez-vous qu'elle s'exécute plus lentement ?
    1. Devriez-vous l'utiliser ? Quand est-ce que cela aurait du sens de le faire ?
    1. À quoi devez-vous faire attention ? Indice : considérez le domaine de définition du logarithme.
1. Est-ce toujours une bonne idée de renvoyer l'étiquette la plus probable ? Par exemple, feriez-vous cela pour un diagnostic médical ? Comment essaieriez-vous de résoudre ce problème ?
1. Supposons que nous voulions utiliser la régression softmax pour prédire le mot suivant sur la base de certaines caractéristiques. Quels sont les problèmes qui pourraient découler d'un vocabulaire étendu ?
1. Expérimentez avec les hyperparamètres du code de cette section. En particulier :
    1. Tracez comment la perte de validation change à mesure que vous modifiez le taux d'apprentissage.
    1. La perte de validation et la perte d'entraînement changent-elles à mesure que vous modifiez la taille du mini-lot ? Jusqu'à quelle taille ou quelle petitesse devez-vous aller avant de voir un effet ?
